In [4]:
import re
import pandas as pd

log_path = "/OpenPCDet/output/OpenPCDet/tools/cfgs/models/S/D/S_D_E/default/eval/epoch_7862/val_all/default/log_eval_20260804-123837.txt"
with open(log_path) as f:
    text = f.read()

pattern = re.compile(
    r'(\w+)\s+(AP_R40@[^:\n]+|AP@[^:\n]+):\n'
    r'bbox AP:([^\n]+)\n'
    r'bev\s+AP:([^\n]+)\n'
    r'3d\s+AP:([^\n]+)',
    re.MULTILINE,
)

def parse_triplet(value_text):
    return [float(x.strip()) for x in value_text.split(',')]

results = {}
rows = []

for match in pattern.finditer(text):
    cls = match.group(1)
    setting = match.group(2)

    bbox_vals = parse_triplet(match.group(3))
    bev_vals = parse_triplet(match.group(4))
    d3_vals = parse_triplet(match.group(5))

    results.setdefault(cls, {})[setting] = {
        'bbox': bbox_vals,
        'bev': bev_vals,
        '3d': d3_vals,
    }

    rows.append({
        'class': cls,
        'setting': setting,
        'metric': 'bbox',
        'easy': bbox_vals[0],
        'moderate': bbox_vals[1],
        'hard': bbox_vals[2],
    })
    rows.append({
        'class': cls,
        'setting': setting,
        'metric': 'bev',
        'easy': bev_vals[0],
        'moderate': bev_vals[1],
        'hard': bev_vals[2],
    })
    rows.append({
        'class': cls,
        'setting': setting,
        'metric': '3d',
        'easy': d3_vals[0],
        'moderate': d3_vals[1],
        'hard': d3_vals[2],
    })

df = pd.DataFrame(rows)
print(df)
print({cls: list(settings.keys()) for cls, settings in results.items()})

         class                  setting metric     easy  moderate     hard
0          Car      AP@0.70, 0.70, 0.70   bbox  74.9207   74.9207  74.9207
1          Car      AP@0.70, 0.70, 0.70    bev  21.4878   21.4878  21.4878
2          Car      AP@0.70, 0.70, 0.70     3d  10.0676   10.0676  10.0676
3          Car  AP_R40@0.70, 0.70, 0.70   bbox  77.5461   77.5461  77.5461
4          Car  AP_R40@0.70, 0.70, 0.70    bev  17.4316   17.4316  17.4316
5          Car  AP_R40@0.70, 0.70, 0.70     3d   3.6105    3.6105   3.6105
6          Car      AP@0.70, 0.50, 0.50   bbox  74.9207   74.9207  74.9207
7          Car      AP@0.70, 0.50, 0.50    bev  37.7956   37.7956  37.7956
8          Car      AP@0.70, 0.50, 0.50     3d  31.6162   31.6162  31.6162
9          Car  AP_R40@0.70, 0.50, 0.50   bbox  77.5461   77.5461  77.5461
10         Car  AP_R40@0.70, 0.50, 0.50    bev  34.6796   34.6796  34.6796
11         Car  AP_R40@0.70, 0.50, 0.50     3d  28.1054   28.1054  28.1054
12  Pedestrian      AP@0.

In [5]:
model = "SECOND Default"
finetuned = "No"
dataset = "nuScenes"
target_setting = "AP_R40@0.70, 0.70, 0.70"


def pick_setting(class_results, preferred_setting):
    if preferred_setting in class_results:
        return preferred_setting

    ap_r40_settings = [setting for setting in class_results if setting.startswith('AP_R40@')]
    if not ap_r40_settings:
        raise KeyError('No AP_R40 settings found for this class.')

    return sorted(ap_r40_settings)[0]


print(results)
print(f"Using setting: {target_setting}")

{'Car': {'AP@0.70, 0.70, 0.70': {'bbox': [74.9207, 74.9207, 74.9207], 'bev': [21.4878, 21.4878, 21.4878], '3d': [10.0676, 10.0676, 10.0676]}, 'AP_R40@0.70, 0.70, 0.70': {'bbox': [77.5461, 77.5461, 77.5461], 'bev': [17.4316, 17.4316, 17.4316], '3d': [3.6105, 3.6105, 3.6105]}, 'AP@0.70, 0.50, 0.50': {'bbox': [74.9207, 74.9207, 74.9207], 'bev': [37.7956, 37.7956, 37.7956], '3d': [31.6162, 31.6162, 31.6162]}, 'AP_R40@0.70, 0.50, 0.50': {'bbox': [77.5461, 77.5461, 77.5461], 'bev': [34.6796, 34.6796, 34.6796], '3d': [28.1054, 28.1054, 28.1054]}}, 'Pedestrian': {'AP@0.50, 0.50, 0.50': {'bbox': [44.6703, 44.6703, 44.6703], 'bev': [10.643, 10.643, 10.643], '3d': [9.0909, 9.0909, 9.0909]}, 'AP_R40@0.50, 0.50, 0.50': {'bbox': [46.3388, 46.3388, 46.3388], 'bev': [2.4976, 2.4976, 2.4976], '3d': [1.5056, 1.5056, 1.5056]}, 'AP@0.50, 0.25, 0.25': {'bbox': [44.6703, 44.6703, 44.6703], 'bev': [41.2329, 41.2329, 41.2329], '3d': [33.8794, 33.8794, 33.8794]}, 'AP_R40@0.50, 0.25, 0.25': {'bbox': [46.3388, 4

In [6]:
cells = []
chosen_setting = target_setting

for cls in ["Car", "Pedestrian", "Cyclist"]:
    class_results = results[cls]
    setting = pick_setting(class_results, chosen_setting)
    bev = class_results[setting]["bev"]
    d3 = class_results[setting]["3d"]

    for i in range(3):
        cells.append(f"{bev[i]:.4f}/{d3[i]:.4f}")

latex_row = (
    f"{model} & {finetuned} & {dataset} & "
    + " & ".join(cells)
    + r" \\")

print(latex_row)

SECOND Default & No & nuScenes & 17.4316/3.6105 & 17.4316/3.6105 & 17.4316/3.6105 & 36.2333/33.8021 & 36.2333/33.8021 & 36.2333/33.8021 & 43.7878/37.2729 & 43.7878/37.2729 & 43.7878/37.2729 \\
